# DuckDB browser

Browse a `.duckdb` file (or any Parquet/CSV) from VS Code using the project's `.venv`.

- Opens the database **read-only**, so nothing here can change it.
- If another process has the file open for writing, the connect cell fails with a lock error: close that process (or its kernel) first.
- Results come back as pandas DataFrames. Click **Open in Data Wrangler** / the Jupyter Variables view for a sortable, filterable grid.

In [18]:
import duckdb
import pandas as pd

from IPython.display import HTML, display

def show(df, rows=300, height=100):
    """Scrollable table: full cell contents, first `rows` rows, fixed-height box."""
    with pd.option_context("display.max_colwidth", None):
        table = df.head(rows).to_html(max_rows=None, max_cols=None)
    note = f"{len(df):,} rows" + (f" (showing first {rows:,})" if len(df) > rows else "")
    display(HTML(f'<p>{note}</p><div style="max-height:{height}px; overflow:auto">{table}</div>'))



pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 500)


DB_PATH = "data/corpus/ledger.duckdb"  # relative to the workspace root

con = duckdb.connect(DB_PATH, read_only=True)
con.sql("SELECT version() AS duckdb_version").df()

,duckdb_version
0,v2.0.0


## Tables and views

In [19]:
con.sql(
    """
    SELECT table_schema, table_name, table_type  -- BASE TABLE or VIEW
    FROM information_schema.tables
    ORDER BY table_schema, table_name
    """
).df()

,table_schema,table_name,table_type
0,main,ledger,BASE TABLE
1,main,mentions,VIEW


## One table: columns and a preview

In [20]:
TABLE = "ledger"  # or "mentions"
# TABLE = "mentions"

con.sql(f"DESCRIBE {TABLE}").df()

,column_name,column_type,null,key,default,extra
0,corpus_filepath,VARCHAR,NO,PRI,None,None
1,corpus_filename,VARCHAR,YES,NaN,None,None
2,hash_id,VARCHAR,YES,NaN,None,None
3,Source,VARCHAR,YES,NaN,None,None
4,Author,"STRUCT(""name"" VARCHAR, ""role"" VARCHAR)[]",YES,NaN,None,None
5,Title,VARCHAR,YES,NaN,None,None
6,CreationDate,DATE,YES,NaN,None,None
7,DocumentDate,DATE,YES,NaN,None,None
8,Region,VARCHAR[],YES,NaN,None,None
9,Country,VARCHAR[],YES,NaN,None,None


In [21]:
show(con.sql(f"FROM {TABLE} LIMIT 200").df())

In [22]:
show(con.sql(
    r"""
    SELECT l.Title, l.Source, l.DocumentDate,
           list_filter(l.Keywords, lambda k: regexp_matches(k, '\bUS\b')) AS matched_keywords,
           l.Keywords AS all_keywords
    FROM ledger l
    WHERE len(list_filter(l.Keywords, lambda k: regexp_matches(k, '\bUS\b'))) > 0
    ORDER BY l.DocumentDate DESC
    """
).df())


,Title,Source,DocumentDate,matched_keywords,all_keywords
0,Slide 1,JP Morgan,2026-11-02,"[Exchange rates vs US$, MSCI US, MSCI US performance, US, US 10Y yield, US Banks, US CPI vs Brent correlation, US Energy Consumption as Share of GDP, US Inflation forwards, US P/E multiple, US Tactical Positioning Monitor, US Treasury RSI, US average hourly earnings, US bank reserves, US employment cost index, US equity performance, US expensive vs World, US fiscal deficit, US gasoline price, US government balances, US government gross debt, US households, US market share, US relative performance, US relative valuation, US sectors, US tariffs on China, US tech capex, US unemployment rate, US-China tariffs]","[% y/y, %6mom, +1 Stdev, -1 Stdev, 10 Yr Govt BY, 10Y spreads to Bunds, 10Y/2Y spread, 10Y/3M spread, 10x forward P/E, 12-month forward EPS, 12M performance, 12m Fw P/E, 12m Fwd EPS, 12m Fwd EPS/Sales, 12m Fwd P/E, 12m Fwd P/E relative, 12m Fwd P/Sales, 12m Fwd PE, 12m Fwd median EPS, 12m forward EPS, 12m forward EPS growth, 12m forward P/E, 12m forward PE, 12m forward PE relative, 12m forward total income, 12m fw EPS Gr, 12m fwd EPS, 12m fwd P/E, 12mth Fwd P/E, 15th Five Year Plan, 20-25% fall, 200 day moving average, 2000 Peak, 2022, 2022 comparison, 2025, 2025 disappointment, 2026, 2026-27, 2026e EPS, 2026e EPS growth, 2026e dividend yield, 2027 budget proposal, 2027e EPS, 20Y median, 25bp hike, 2H broadening, 2Std, 3-month lookback, 3M performance, 3Y moving average, 3Y moving average less 2 std, 45% discount to DM, 5-yr avg, 5-yr range, 55bp+ by Q4 next year, 60bp of Fed cuts, 70-city average house prices, 76% abroad exposure, AAII Bull Bear, AAII Bull-Bear, AI, AI Infrastructure, AI adoption, AI at risk basket, AI cannibalisation, AI cannibalization, AI capex, AI datacenter electrification, AI demand, AI disruption, AI headwinds, AI indirect plays, AI monetization concerns, AI related derating, AI uncertainty, AI unwind risks, AUM, Accor SA, Aerospace & Defense, Ageing population, Asia AI Basket, Atlanta Fed Wage Growth Tracker, Autos margins, BANK (XET), BE Semiconductor Industries, BEA NIPA, Bank of France debt holdings, Banks, Banks and Energy, Beer vs Spirits, Beneficiaries of greenshoots in China, Bloomberg, Bloomberg Finance, Bloomberg Finance L.P., Blue Wave, Bond yields, Brent stabilization, Brent ytd, Business Services, ...]"
1,NaN,BNP Paribas,2026-11-01,[US midterms],"[AI trade correction, Brazil fiscal risk premium, Christine Lagarde, Democratic sweep, ECB succession, FX hedge ratios, Klaas Knot, Middle East escalation, Pablo Hernández de Cos, RBA hawkish rhetoric, Strait of Hormuz, US midterms, anti-AI sentiment, blue sweep, cold weather, earnings season, energy price shock, fiscal risk, fiscal spending, hedge, low wind, probes and subpoenas, receive the meeting, risk event, second-round effects, short 30y swap spread positions, short USD vs CHF and GBP, wage pressures]"
2,NaN,BNP Paribas,2026-11-01,"[US Equity Derivatives Strategy, US Equity Positioning Indicator, US equity positioning]","[$250bn loss, -35.3bn, 10d RV, 1m ATM IV, 1m RV, 200d MA, 200d moving average, 2m ATM IV, 2yr valuation low, 3m post-election, 3m pre-election, 6m trading range, 84% EPS beats, AI bull market, AI vs. the rest, BNP Paribas, BNP Paribas Securities Corp, Black Monday, Bloomberg, Brazilian market, CFTC, CTA, Call Fly, Call Ratio, Call Spread, Dec26 upside, DeepSeek, Democratic sweep, EPI, EWZ, Friday sell-off, IWM Nov26 ideas menu, LETF rebalance outflows, Lev. ETF Daily Rebalance Flow, Max PnL, Nov26 740/725 Put Spread, Nov26 Upside, Nov26 expiry, Nov26 ideas menu, Oct-30-2026, Premium Delta Discount Payout, Put Ratio, Q3 earnings led rally, RTY-SPX volatility, SPX 1m ATM IV, SPX 1m RV, SPX buybacks, SPX implied volatility, SPX returns, SPY put spreads, Thanksgiving Holiday, US Equity Derivatives Strategy, US Equity Positioning Indicator, US equity positioning, VT sales, Vol Target, buybacks, call fly, call ratio, call spread, de-corre

In [23]:
# Per-column stats: min/max, distinct count, null %
con.sql(f"SUMMARIZE {TABLE}").df()

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,corpus_filepath,VARCHAR,bnp/2025_EUR_government_bond_and_SSA_issuance_outlook_Desknote--850446.pdf,statestreet/Mind_and_Machine_FX/d5271797-4286-4181-9fb0-a880b54a32df.pdf,11247,NaN,None,NaN,NaN,NaN,10682,0.00
1,corpus_filename,VARCHAR,00543cdd-5d1d-4133-94c0-918c544a6940.pdf,ffdb56c9-b567-4db8-bf94-d49b21d08f98.pdf,10248,NaN,None,NaN,NaN,NaN,10682,0.00
2,hash_id,VARCHAR,0002e2fba2b5adc7d0b001cac7778cc8216407420adccd10dec5d9c09c901139,ffff6226c3bda54bf8f84a688e1946ac6dc4c847d94e67844a4e3f3c77ceb173,10285,NaN,None,NaN,NaN,NaN,10682,0.00
3,Source,VARCHAR,BNP Paribas,UBS,12,NaN,None,NaN,NaN,NaN,10682,5.57
4,Author,"STRUCT(""name"" VARCHAR, ""role"" VARCHAR)[]",[],"[{'name': Yuting Yang, 'role': NULL}]",2700,NaN,None,NaN,NaN,NaN,10682,0.00
5,Title,VARCHAR,(1) Strait of Hormuz: A Citrini Field Trip - Citrini Research,metal&ROCK: Why Is Aluminium Not Higher?,5319,NaN,None,NaN,NaN,NaN,10682,42.17
6,CreationDate,DATE,2023-09-14,2026-11-02,692,2025-10-15 12:20:29.432691,None,2025-03-31,2025-10-18,2026-04-24,10682,0.00
7,DocumentDate,DATE,2024-07-02,2026-11-02,709,2025-10-19 14:09:16.48038,None,2025-03-31,2025-10-23,2026-04-30,10682,5.52
8,Region,VARCHAR[],[],[NA],64,NaN,None,NaN,NaN,NaN,10682,0.00
9,Country,VARCHAR[],[],[ZA],3369,NaN,None,NaN,NaN,NaN,10682,0.00


## Ad-hoc SQL

Also works on files directly, e.g. `FROM 'outputs/some_file.parquet' LIMIT 100` or `FROM read_csv('path.csv')`.

In [24]:
con.sql(
    """
    SELECT Source, count(*) AS docs, min(DocumentDate) AS first, max(DocumentDate) AS last
    FROM ledger
    GROUP BY Source
    ORDER BY docs DESC
    """
).df()

,Source,docs,first,last
0,BNP Paribas,2613,2024-07-02,2026-11-02
1,Goldman Sachs,1199,2024-12-29,2026-10-31
2,Morgan Stanley,1098,2025-01-01,2026-10-30
3,Deutsche Bank,952,2024-12-30,2026-10-30
4,UBS,790,2025-01-02,2026-10-30
5,Barclays,746,2025-01-01,2026-10-30
6,Citi,719,2025-01-02,2026-10-30
7,State Street,710,2024-07-25,2026-10-30
8,Bank of America,625,2024-12-13,2026-10-30
9,JP Morgan,597,2024-12-06,2026-11-02


In [25]:
# Release the file so other processes can write to it again
con.close()